In [1]:
#!pip install --trusted-host pypi.org --trusted-host files.pythonhosted.org scikit-posthocs

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from scipy import stats
import matplotlib.ticker as mticker

# ==========================================
# 1. Configuration and Paths
# ==========================================

# Base project directory
BASE_PATH = 'E:\\Projetos\\ABM-WP' # Currently active path

# Input Files
SCENARIO_FILE = 'consumo_previsto_todos_cenarios.csv'
LSTM_FILE = 'previsoes_futuras_2025_2035.csv'

# Output File
OUTPUT_FIG = 'normality_analysis_scenarios_vs_lstm.pdf'

# Directory Setup
INPUT_DIR_RES = os.path.join(BASE_PATH, 'resultados')
INPUT_DIR_MODELS = os.path.join(BASE_PATH, 'modelos IA')
INPUT_DIR_INC = os.path.join(BASE_PATH, 'includes')
OUTPUT_DIR = os.path.join(BASE_PATH, 'figuras')

# Style configuration
plt.style.use('seaborn-v0_8-white')
sns.set_palette("husl")

def sum_yearly(df, date_col, value_cols):
    """ Helper function to aggregate monthly data into annual sums """
    df_year = df.copy()
    df_year['Year'] = df_year[date_col].dt.year
    return df_year.groupby('Year')[value_cols].sum().reset_index()

def main():
    print("--- Starting Normality Analysis: 18 Scenarios vs LSTM ---")

    # ==========================================
    # 2. Load and Prepare Data
    # ==========================================
    
    # Load Scenario Data (18 scenarios)
    path_scenarios = os.path.join(INPUT_DIR_RES, SCENARIO_FILE)
    if not os.path.exists(path_scenarios):
        path_scenarios = os.path.join(INPUT_DIR_INC, SCENARIO_FILE)
    
    # Using semicolon separator and comma for decimals as per the 18-scenario file format
    df_scenarios = pd.read_csv(path_scenarios, sep=';', decimal=',')
    
    # Date Construction
    if 'Mes' in df_scenarios.columns and 'Ano' in df_scenarios.columns:
        df_scenarios['Date'] = pd.to_datetime(df_scenarios['Ano'].astype(str) + '-' + 
                                            df_scenarios['Mes'].astype(str) + '-01')
    elif 'Mes_Ano' in df_scenarios.columns:
        df_scenarios['Date'] = pd.to_datetime(df_scenarios['Mes_Ano'], format='%m/%Y')
    
    # Identify Scenario Columns
    metadata_cols = ['Mes', 'Ano', 'Mes_Ano', 'Date']
    scenario_cols = [c for c in df_scenarios.columns if c not in metadata_cols]
    df_scenarios = df_scenarios[df_scenarios['Date'] <= pd.to_datetime('2034-12-31')]

    # Load LSTM Data
    path_lstm = os.path.join(INPUT_DIR_MODELS, LSTM_FILE)
    df_lstm = pd.read_csv(path_lstm, parse_dates=['Data'])
    df_lstm = df_lstm.rename(columns={'Data': 'Date', 'Valor Previsto': 'LSTM'})
    df_lstm = df_lstm[df_lstm['Date'] <= pd.to_datetime('2034-12-31')]

    # Merge and Aggregate
    df_combined = pd.merge(df_scenarios, df_lstm[['Date', 'LSTM']], on='Date', how='left')
    all_series = scenario_cols + ['LSTM']
    df_annual = sum_yearly(df_combined, 'Date', all_series)

    # ==========================================
    # 3. Graphical Analysis (Distribution & Q-Q Plots)
    # ==========================================
    
    # Save all plots in a single PDF with multiple pages
    from matplotlib.backends.backend_pdf import PdfPages
    
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
    
    output_path = os.path.join(OUTPUT_DIR, OUTPUT_FIG)
    
    with PdfPages(output_path) as pdf:
        
        # --- Page 1: Histograms / KDE ---
        plt.figure(figsize=(14, 8))
        for col in all_series:
            # Use Scenario IDs for labels
            label_id = col.split('_')[0] if '_' in col else col
            sns.histplot(df_annual[col], kde=True, label=label_id, element="step", alpha=0.1)
            
        plt.title('Distribution Comparison: 18 Scenarios and LSTM Model', fontsize=16, fontweight='bold')
        plt.xlabel('Annual Consumption (m³)', fontsize=12)
        plt.ylabel('Frequency', fontsize=12)
        plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x/1000:.0f}K'))
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8, ncol=1)
        plt.tight_layout()
        pdf.savefig()
        plt.close()

        # --- Page 2 & 3: Q-Q Plots (Grids) ---
        # Splitting into two pages to maintain readability (19 total series)
        series_chunks = [all_series[i:i + 10] for i in range(0, len(all_series), 10)]
        
        for chunk in series_chunks:
            plt.figure(figsize=(16, 12))
            for i, scenario in enumerate(chunk, 1):
                plt.subplot(4, 3, i)
                label_id = scenario.split('_')[0] if '_' in scenario else scenario
                stats.probplot(df_annual[scenario], plot=plt)
                plt.title(f'Q-Q Plot: {label_id}', fontsize=10)
                plt.grid(True, alpha=0.3)
            plt.tight_layout()
            pdf.savefig()
            plt.close()

    # ==========================================
    # 4. Statistical Normality Tests (Shapiro-Wilk)
    # ==========================================
    
    normality_results = []
    for scenario in all_series:
        label_id = scenario.split('_')[0] if '_' in scenario else scenario
        stat, p = stats.shapiro(df_annual[scenario])
        normality_results.append({
            'ID': label_id,
            'Statistic W': stat,
            'p-value': p,
            'Normal?': 'Yes' if p > 0.05 else 'No'
        })

    normality_df = pd.DataFrame(normality_results)
    
    print("\n--- Shapiro-Wilk Normality Test Results ---")
    print(normality_df.round(4).to_string(index=False))
    
    # Save the table results to a CSV for documentation
    table_output = os.path.join(INPUT_DIR_RES, 'normality_test_results.csv')
    normality_df.to_csv(table_output, index=False, sep=';')
    print(f"\nAnalysis complete. Plot saved to: {output_path}")


if __name__ == "__main__":
    main()

--- Starting Normality Analysis: 18 Scenarios vs LSTM ---

--- Shapiro-Wilk Normality Test Results ---
    ID  Statistic W  p-value Normal?
    CI       0.9356   0.5052     Yes
   CII       0.9356   0.5052     Yes
  CIII       0.9356   0.5052     Yes
   CIV       0.9583   0.7666     Yes
    CV       0.9585   0.7687     Yes
   CVI       0.9587   0.7709     Yes
  CVII       0.9583   0.7668     Yes
 CVIII       0.9585   0.7689     Yes
   CIX       0.9587   0.7711     Yes
    CX       0.9582   0.7649     Yes
   CXI       0.9583   0.7667     Yes
  CXII       0.9585   0.7684     Yes
 CXIII       0.9584   0.7671     Yes
  CXIV       0.9586   0.7693     Yes
   CXV       0.9588   0.7717     Yes
  CXVI       0.8935   0.1857     Yes
 CXVII       0.9136   0.3069     Yes
CXVIII       0.9370   0.5205     Yes
  LSTM       0.6207   0.0001      No

Analysis complete. Plot saved to: E:\Projetos\ABMS-WP\figuras\normality_analysis_scenarios_vs_lstm.pdf
